# Project Nano-Myo
## Notebook 03 - Baseline Training


## Step 1 - Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 - Imports


In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Nano_Myo'
FEATURE_DIR = PROJECT_ROOT / 'features'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SCHEMA_VERSION = 'e2_only_80feat_wl_ssc_rest25000_v1'
INPUT_DIM = 80
HIDDEN_UNITS = (64, 32)
NUM_CLASSES = 10

MODEL_PATH = MODEL_DIR / 'baseline_float32.keras'
SCALER_PATH = MODEL_DIR / 'scaler.pkl'
RESULTS_PATH = MODEL_DIR / 'baseline_results.txt'


## Step 3 - Load Artifacts


In [ ]:
paths = {
    'X_train': FEATURE_DIR / f'X_train_{FEATURE_SCHEMA_VERSION}.npy',
    'y_train': FEATURE_DIR / f'y_train_{FEATURE_SCHEMA_VERSION}.npy',
    'X_test': FEATURE_DIR / f'X_test_{FEATURE_SCHEMA_VERSION}.npy',
    'y_test': FEATURE_DIR / f'y_test_{FEATURE_SCHEMA_VERSION}.npy',
    'metadata': FEATURE_DIR / f'nano_myo_features_holdout_s9_s10_{FEATURE_SCHEMA_VERSION}_metadata.json',
}

missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

with open(paths['metadata'], 'r', encoding='utf-8') as f:
    metadata = json.load(f)

if metadata['feature_schema_version'] != FEATURE_SCHEMA_VERSION:
    raise ValueError(metadata['feature_schema_version'])
if metadata['exercise_used'] != 'E2':
    raise ValueError(metadata['exercise_used'])
if metadata['input_dim'] != INPUT_DIM:
    raise ValueError(metadata['input_dim'])
if set(metadata['feature_groups']) != {'mav', 'zc', 'rms', 'wl', 'ssc'}:
    raise ValueError(metadata['feature_groups'])

X_train = np.load(paths['X_train']).astype(np.float32)
y_train = np.load(paths['y_train']).astype(np.int64)
X_test = np.load(paths['X_test']).astype(np.float32)
y_test = np.load(paths['y_test']).astype(np.int64)

if X_train.shape[1] != INPUT_DIM or X_test.shape[1] != INPUT_DIM:
    raise ValueError((X_train.shape, X_test.shape))

TARGET_LABELS = {int(key): value for key, value in metadata['target_labels'].items()}

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'Train counts: {Counter(y_train.tolist())}')
print(f'Test counts: {Counter(y_test.tolist())}')


## Step 4 - Model


In [ ]:
def build_model(input_dim=INPUT_DIM, hidden_units=HIDDEN_UNITS, num_classes=NUM_CLASSES):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(hidden_units[0], activation='relu')(inputs)
    x = tf.keras.layers.Dense(hidden_units[1], activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)


model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

param_count = model.count_params()
model.summary()
print(f'Parameter count: {param_count:,}')


## Step 5 - Train


In [ ]:
classes = np.arange(NUM_CLASSES)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight = {int(class_id): float(weight) for class_id, weight in zip(classes, weights)}

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=50,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)


## Step 6 - Evaluate


In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_prob = model.predict(X_test, batch_size=256, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

per_class_accuracy = {}
for class_id, class_name in TARGET_LABELS.items():
    mask = y_test == class_id
    per_class_accuracy[class_id] = float(np.mean(y_pred[mask] == y_test[mask])) if np.any(mask) else float('nan')
    print(f'{class_id}: {class_name}: {per_class_accuracy[class_id]:.4f}')

print(f'Test accuracy: {test_accuracy:.4f}')


## Step 7 - Confusion Matrix


In [ ]:
labels = list(TARGET_LABELS.keys())
display_labels = [TARGET_LABELS[key] for key in labels]
cm = confusion_matrix(y_test, y_pred, labels=labels, normalize='true')

fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm, display_labels=display_labels).plot(
    include_values=True,
    cmap='Blues',
    ax=ax,
    xticks_rotation=45,
    values_format='.2f',
)
plt.title('Baseline Float32 Confusion Matrix')
plt.tight_layout()
plt.show()


## Step 8 - Save


In [ ]:
model.save(MODEL_PATH)

normalization_info = {
    'method': 'per_subject_standardization',
    'source': 'Notebook 02',
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'saved_arrays_are_already_normalized': True,
    'uses_labels': False,
}
joblib.dump(normalization_info, SCALER_PATH)

lines = [
    'Project Nano-Myo baseline float32 results',
    f'feature_schema_version: {FEATURE_SCHEMA_VERSION}',
    f'input_dim: {INPUT_DIM}',
    f'hidden_units: {HIDDEN_UNITS}',
    f'parameter_count: {param_count}',
    f'test_accuracy: {test_accuracy:.6f}',
    '',
    'per_class_accuracy:',
]

for class_id, class_name in TARGET_LABELS.items():
    lines.append(f'{class_id}: {class_name}: {per_class_accuracy[class_id]:.6f}')

RESULTS_PATH.write_text('\n'.join(lines), encoding='utf-8')

print(f'Model saved: {MODEL_PATH}')
print(f'Normalization metadata saved: {SCALER_PATH}')
print(f'Results saved: {RESULTS_PATH}')
